# Run Model

Thin, restart-safe runner for one country and scenario. Configure the inputs, run the workflow, and inspect the manifest. Standard plots and exploratory analysis live in the companion notebooks.

## Companion notebooks

- `run_model_diagnostics.ipynb` — standard macro and model diagnostics
- `run_model_exploration.ipynb` — exploratory firm, household, reader, and ratio analysis
- `run_sensitivity.ipynb` — parameter sensitivity
- `run_mpc.ipynb` — household MPC experiment
- `run_irf.ipynb` — macro impulse responses
- `run_model_legacy_2026-08-18.ipynb` — preserved historical notebook; do not use for new work

In [ ]:
# Setup
%load_ext autoreload
%autoreload 2

import numpy as np

from src.monte_carlo import run_seeded_monte_carlo
from src.notebook_config import MACRO_COLUMNS, SCENARIO_PRESETS
from src.notebook_state import run_notebook_workflow, validate_notebook_state
from src.notebook_workflow import NotebookRunConfig
from src.visual_helpers import plot_mc


## Inputs

All execution switches are centralized here. Sensitivity, MPC, and IRF switches are in their dedicated notebooks.

In [ ]:
RUN_BENCHMARK = True
RUN_MONTE_CARLO = False
SCENARIO_NAME = "calibrated_consumption"

run_config = NotebookRunConfig(
    seed=232,
    t_max=150,
    country_iso3="FRA",
    run_benchmark=RUN_BENCHMARK,
    force_rebuild_data=True,
    force_rerun_benchmark=True,
    benchmark_overrides=None,
)
scenario_overrides = SCENARIO_PRESETS[SCENARIO_NAME]


## Run simulation

In [ ]:
state = run_notebook_workflow(
    run_config,
    scenario_name=SCENARIO_NAME,
    scenario_overrides=scenario_overrides,
)

# Familiar aliases; `state` remains the authoritative carrier.
COUNTRY = state.country_code
prepared = state.prepared
data = prepared.data
cfg = prepared.cfg
simulation = state.simulation
model = state.model
df_base = state.df_base
benchmark = state.benchmark
df_benchmark = state.df_benchmark

validate_notebook_state(state)


In [ ]:
{
    "country": COUNTRY,
    "seed": cfg.seed,
    "t_max": cfg.t_max,
    "scenario": state.scenario_name,
    "model_h5": str(simulation.model_h5_path),
    "benchmark": benchmark is not None,
    "manifest": str(state.manifest_path),
}


## Optional Monte Carlo

In [ ]:
if RUN_MONTE_CARLO:
    rng = np.random.default_rng(run_config.seed)
    mc_seeds = rng.choice(np.arange(1000), size=50, replace=False).tolist()
    mc = run_seeded_monte_carlo(
        datawrapper=data,
        country_configurations=state.country_configurations,
        country_code=COUNTRY,
        seeds=mc_seeds,
        t_max=cfg.t_max,
        n_jobs=-1,
        backend="loky",
        batch_size=1,
    )
    plot_mc(mc=mc, cols=list(MACRO_COLUMNS), no_cols=4, country_code=COUNTRY)
else:
    print("Set RUN_MONTE_CARLO = True to run seeded Monte Carlo simulations.")
